# Jet Bigrams — GPT-2 Large

Evaluates the jet bigram expansions over the full vocabulary:

- **E → U**: pure embedding-unembedding path (`embedding_unembedding`)
- **E → MLP$_l$ → U**: embedding through one MLP layer then unembedding (`embedding_mlp_unembedding`)

For each input token $z_1$, the bigram path computes the conditional likelihood of token $z_2$

In [1]:
import torch
import jex.models as models
from jex.ngrams.bigrams import embedding_unembedding, embedding_mlp_unembedding
from jex.ngrams.utils import eval_over_vocab, top_bigrams, print_bigram_tables_side_by_side

## Load model

In [2]:
lm = models.from_pretrained("gpt2-large")
print(f"Loaded {lm.name}: {lm.depth} layers, vocab size {lm.vocab_size}")

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

Loaded gpt2-large: 36 layers, vocab size 50257


## Evaluate over the full vocabulary

We run each bigram expansion on every token in the vocabulary (in batches) and collect the full `(vocab, vocab)` logit matrix.

In [3]:
n = 100

jet_eu = embedding_unembedding(lm)
with torch.no_grad():
    topk_eu = top_bigrams(eval_over_vocab(jet_eu, lm), lm, n=n)
print("E→U done")

E→U done


In [ ]:
topk_mlp = {}
for l in range(lm.depth):
    jet = embedding_mlp_unembedding(lm, l)
    with torch.no_grad():
        topk_mlp[l] = top_bigrams(eval_over_vocab(jet, lm), lm, n=n)
    print(f"layer {l} done", end="\r")
print(f"All {lm.depth} MLP layers done.")

## Top-100 bigram pairs

Global top-100 (input, next) pairs by logit score, shown for three paths side by side.

In [ ]:
print_bigram_tables_side_by_side([
    ("E → U", topk_eu),
    *[(f"E → MLP_{l} → U", rows) for l, rows in topk_mlp.items()]
])